# Biomechanical RL Course: MyoInteract UI — Google Colab Notebook

Run the cells below in order.  
**Step 1** installs all dependencies.  
**Step 2** launches the Gradio configuration UI.  
**Step 3** starts training with the settings you chose in the UI.

> **GPU required.** This notebook needs a CUDA-capable GPU for mjlab/Warp training.  
> In Google Colab: *Runtime → Change runtime type → T4 GPU* (or better).

In [ ]:
# ── Install all dependencies ────────────────────────────────────────────────
# Run once per runtime.  Takes ~3 minutes on a fresh Colab instance.

# Core simulation + RL stack
!pip install --quiet \
    "mujoco>=3.2" \
    "mujoco-warp>=3.7.0" \
    "warp-lang==1.12.0" \
    "mjlab==1.3.0" \
    "scipy" \
    "onnxruntime"

# Viewer (pinned for viser API compatibility)
!pip install --quiet \
    "viser==1.0.26" \
    "mjviser==0.0.12"

# UI + logging
!pip install --quiet \
    "gradio>=5.0" \
    "gradio-rangeslider==0.0.8" \
    "wandb>=0.21.0"

# Config / utilities
!pip install --quiet \
    "ml-collections>=0.1.1" \
    "hydra-core" \
    "omegaconf" \
    "etils[epath]>=1.3.0"

# ── Install this package from GitHub ────────────────────────────────────────
# Replace the URL below with your own fork if needed.
!pip install --quiet \
    "git+https://github.com/fl0fischer/myointeract_ui_course.git"

import importlib.util, sys
from pathlib import Path

# viser_viewer_ext lives alongside gradio_ui in the installed package;
# adding its directory to sys.path lets gradio_ui import it without a
# package prefix (matching the original relative-import convention).
_spec = importlib.util.find_spec("tutorials.myouser.gradio_ui")
if _spec and _spec.origin:
    _ui_dir = str(Path(_spec.origin).parent)
    if _ui_dir not in sys.path:
        sys.path.insert(0, _ui_dir)

# ── Verify GPU access (Google Colab) ────────────────────────────────────────
# Dependencies like mujoco-warp or mjlab can silently upgrade PyTorch to a
# build compiled for a newer CUDA toolkit than what Colab's driver supports,
# making torch.cuda.is_available() return False and causing training to freeze
# on CPU.  Detect this and reinstall a compatible build automatically.
import os as _os
_IN_COLAB = 'google.colab' in sys.modules or _os.path.exists('/content')
if _IN_COLAB:
    import subprocess as _sp, torch as _torch
    if not _torch.cuda.is_available():
        print(
            f"[setup] torch {_torch.__version__} cannot access the GPU — "
            "re-installing from the cu124 wheel index …"
        )
        _sp.check_call([
            sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
            'torch', 'torchvision',
            '--index-url', 'https://download.pytorch.org/whl/cu124',
        ])
        print(
            "[setup] PyTorch reinstalled. "
            "Restart the runtime (Runtime ▸ Restart runtime), then re-run from cell 2."
        )
        raise SystemExit
    print(f"[setup] GPU ready: {_torch.cuda.get_device_name(0)}")

print(f'Installation complete.')
print('Restart the runtime if prompted, then re-run from the next cell.')

In [1]:
## Enter your personal project name. Used to group runs on WandB.
PROJECT_NAME = "test-123"

## Your WandB entity (username or team name). Leave as None to use your default entity.
WANDB_ENTITY = None  # e.g. "my-username" or "my-team"

# Uncomment and fill in to log into WandB without interactive prompt:
# import wandb
# wandb.login(key="YOUR_WANDB_API_KEY")

In [2]:
import wandb
from gradio_ui import get_ui, RunState

state = RunState()
demo = get_ui(PROJECT_NAME, state)
demo.launch(share=True, height=800)

/home/mifs/fjf33/scratch/mamba/envs/course_ui_colab_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


╭────── viser (listening *:7861) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:7861   │
│   Websocket │ ws://localhost:7861     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://intrinsic-biped.share.viser.studio

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fc909ca667c5f27163.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/scratch/fjf33/mamba/envs/course_ui_colab_test/lib/python3.12/site-packages/myosuite/envs/myo/assets/arm/mobl_arms_index_universal_myouser.xml
/scratch/fjf33/mamba/envs/course_ui_colab_test/lib/python3.12/site-packages/myosuite/envs/myo/assets/arm/mobl_arms_index_universal_myouser.xml
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 12.8
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5090" (31 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/mifs/fjf33/.cache/warp/1.12.0
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 1.60 ms  (cached)
Module mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 1.14 ms  (cached)
Module _nxn_broadphase__locals__kernel_e73b1315 e73b131 load on device 'cuda:0' took 1.00 m

(viser) Connection opened (0, 1 total), 311 persistent messages

---
After configuring the task in the UI above, execute the cell below to start training:

In [ ]:
from gradio_ui import launch_training

wandb.init(project=PROJECT_NAME, name=state.wandb_run_name, entity=WANDB_ENTITY)
launch_training(state)

Please fill in this form to provide feedback on the UI:

https://forms.gle/9P2zHqPckMnomcJS9